In [ ]:
from sympy import *
import numpy as np

In [ ]:
n, eps, r = symbols("n, epsilon, r")

In [ ]:
MIR = 0.0125*n#-1/2*log(1-(n*eps)**2)
Delta = simplify(diff(MIR, n))
Sigma = simplify(diff(Delta, n)/2)
display(MIR, Delta, Sigma)

In [ ]:
S1, G1, T, d, A, B = symbols("S', G', T, delta, A, B")

In [ ]:
S_m = MIR + Sigma - Delta
S_p = MIR + Sigma + Delta
display(S_m, S_p)

In [ ]:
S_m1 = S1 + A*(Sigma - Delta) - B/2*(Sigma - Delta)**2
S_p1 = S1 + A*(Sigma + Delta) - B/2*(Sigma + Delta)**2
lower = (S1- S_m1)/2
upper = (S_p1 - S1)/2
basin = upper + lower
display(simplify(basin), Sigma.n(n=4, subs={"n":186, "epsilon":0.005}), Delta.n(n=4, subs={"n":186, "epsilon":0.005}))
display(S_m1.n(n=4, subs={"n":186, "epsilon":0.005, A:0.95, B:0.115}), S_p1.n(n=4, subs={"n":186, "epsilon":0.005, A:0.95, B:0.115}))

In [ ]:
# T1 = G1 - A*d - B/2*d**2
T1 = G1 + A*d - B/2*d**2
display(T1)

In [ ]:
# EV = simplify((Delta - Sigma)*(G1-T1)/basin)
EV = simplify((Delta + Sigma)*(T1-G1)/basin)
EV

In [ ]:
display((Delta + Sigma).n(n=4, subs={"n":186, "epsilon":0.005}), (T1-G1).n(n=4, subs={d:0.03, A:0.95, B:0.115}), basin.n(n=4, subs={"n":186, "epsilon":0.005, A:0.95, B:0.115}))

In [ ]:
display((EV/d).evalf(n=4, subs={"n":186, "epsilon":0.005, d:0.03, A:0.95}))
tmp = simplify((EV/d).evalf(n=6, subs={"epsilon":0.005}))
tmp

In [ ]:
tomap = lambdify((A,B,n,d), tmp)

maxd = lambdify((A,B,n), basin.n(n=6, subs={"epsilon":0.005}))

In [ ]:
tomap(0.96, 0.113, 186, 0.01)
maxd(0.96, 0.113, 198)

In [ ]:
#%matplotlib notebook
import matplotlib.pyplot as plt

a=0.96
b=0.113
x=np.zeros((200,200))
y=np.zeros_like(x)
_y=0.00005*np.power(1.0423421498355023, np.arange(200))
z=np.full_like(x, np.nan)
for j in range(200):
    topd=maxd(a, b, j)
    for i in range(200):
        x[j,i]=j
        y[j,i]=_y[i]
        if _y[i]>topd:
            break
        z[j,i]=tomap(a, b, j, _y[i])
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_wireframe(x[:,:100],y[:,:100],z[:,:100])#, rstride = 1, cstride = 1)
#ax.plot_wireframe(x,y,np.ones_like(z), color="orange",rcount=10, ccount=10)
plt.show()

In [ ]:
plt.plot(x[:,0],z[:,0])
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import axes3d
X, Y, Z = axes3d.get_test_data(0.05)


In [ ]:
np.exp(np.log(4000)/200)

In [ ]:
from scipy.stats import pearsonr
from tqdm.notebook import tqdm
increment = 0.005
i=10
means = 0, 0
correlations = [[1, i * increment], [i * increment, 1]]
n_samples = 240
n_iterations = 50000
effective = np.empty(n_iterations)
for j in tqdm(range(n_iterations)):
    points = np.random.multivariate_normal(means, correlations, n_samples).T.copy()
    effective[j] = pearsonr(*points)[0]
myBins=np.arange(0,0.2,0.005)-0.0025
myBins[0]=0
val = plt.hist(np.abs(effective), bins=myBins)
val[2][10].set_color("r")
plt.plot([0.05,0.05],[500,2000], ":k")
plt.title(f"{100*val[0][10]/n_iterations:.4} %")
plt.show()

In [ ]:
myBins=np.arange(0,0.2,0.005)-0.0025
myBins[0]=0
val = plt.hist(np.abs(effective), bins=myBins)
val[2][10].set_color("r")
plt.plot([0.05,0.05],[500,2000], ":k")
plt.show()

In [ ]:
myBins=np.linspace(0,1,200+1, endpoint=True)-0.005/2
myBins[0]=0

twoBins=np.linspace(0,1,200+1, endpoint=True)
twoBins[1:]-=np.diff(twoBins)/2

myBins-twoBins, np.linspace(0,1,200+1, endpoint=True)
binned = pd.cut(all_correcting["r"], myBins)
def myfunc (df):
    return pd.DataFrame([[df["MI"].mean(), df["MI"].std(), df["r"].mean(), df["r"].count(), df.name.mid]], columns=["MI", "std", "<r>", "#","r"])
results = all_correcting.groupby(binned).apply(myfunc)
display(results)

In [ ]:
import pandas as pd
from support import single_iter
import multiprocessing as mp
from tqdm.notebook import tqdm
import numpy as np
steps=70
workers = 12
nsamples = 240
nbins = 8
iters = 50000
incre = 1 / steps
correction = np.zeros(steps)
trueval = np.zeros(steps)
pool = mp.Pool(workers)
correcting = []
for i in tqdm(range(steps), "Computing correction"):
    means = 0, 0
    corre = [[1, newrs[i]], [newrs[i], 1]]
    I = pool.map(
        single_iter,
        (
            (means, corre, nsamples, nbins)
            for __ in range(iters)
        ),
    )
    correcting.append(pd.DataFrame(np.array(I), columns=["MI","r"]))
    trueval[i] = -0.5 * np.log(1 - (newrs[i]) ** 2)

In [ ]:

all_correcting = pd.concat(correcting)
myBins=np.zeros(len(newrs)+1)
myBins[1:]=newrs
myBins[1:-1]+=np.diff(newrs)/2
# myBins=np.linspace(0,1,steps+1)-incre/2
# myBins[0]=0
all_correcting["r"]=all_correcting["r"].abs()
binned = pd.cut(all_correcting["r"], myBins)
def myfunc (df):
    return pd.DataFrame([[df["MI"].mean(), df["MI"].std(), df["r"].mean(), df["r"].count(), df.name.mid]], columns=["MI", "stdMI", "<r>", "#","r"])
results = all_correcting.groupby(binned).apply(myfunc)
display(results.dropna())

In [ ]:
plt.plot(trueval,(results["std"]/results.MI).to_numpy()*100)

# plt.plot(trueval,1/trueval)

In [ ]:
import matplotlib.pyplot as plt

correction = results["MI"].to_numpy()
weights = np.zeros_like(trueval)
weights[:-1] += 0.5 * (trueval[1:] - trueval[:-1])
weights[1:] += weights[:-1]
deviation = np.sqrt(
    np.average(np.square(correction[:] - trueval[:]), weights=weights)
)


tosmo = np.array(
    [
        correction[0],
    ]
    * 2
    + correction.tolist()
    + [
        correction[-1],
    ]
    * 2
)
newco = np.zeros_like(correction)
for i in range(len(correction)):
    newco[i] = np.mean(tosmo[i : i + 5])
plt.plot(correction[:50])
plt.plot(newco[:50])
plt.show()


plt.title(f"RMS correction: {deviation:.4}")
plt.plot(trueval, correction)
plt.plot(trueval, newco)
plt.plot(
    [min(trueval), max(trueval)],
    [min(trueval), max(trueval)],
    ":k",
)
plt.xlabel("True MI")
plt.ylabel("Estimated MI")

In [ ]:
old_newco = np.load("/home/tani/Documents/nonLinear/MS_fMRI_bin8/newco_8.npy")
old_trueval = np.load("/home/tani/Documents/nonLinear/MS_fMRI_bin8/trueval_8.npy")


plt.title(f"RMS correction: {deviation:.4}")
plt.plot(trueval, correction, label="corr")
plt.plot(trueval, newco, label="newc")
plt.plot(old_trueval, old_newco, "--", label="old_newc")
plt.plot(
    [min(trueval), max(trueval)],
    [min(trueval), max(trueval)],
    ":k",
)
plt.legend()
plt.xlabel("True MI")
plt.ylabel("Estimated MI")
plt.show()

plt.plot(old_trueval[:33],old_newco[:33], marker="x")
plt.plot(trueval[:5],correction[:5], marker="^")
plt.plot(mid_trueval[:13],mid_correction[:13], marker="+")
v=np.polyfit(mid_trueval[4:11],mid_correction[4:11],2)
print(v)
def quadint(x, v):
    return v[0]*x**2+v[1]*x+v[2]
plt.plot(mid_trueval[:10],quadint(mid_trueval[:10],v), ":k", marker="3")
plt.xlabel("True MI")
plt.ylabel(r"$\Delta$ MI")
plt.show()
plt.plot(old_trueval[-17:],old_newco[-17:], marker="x")
plt.plot(mid_trueval[-100:],mid_correction[-100:], marker="+")
plt.xlabel("True MI")
plt.ylabel(r"$\Delta$ MI")
plt.show()

In [ ]:
old_trueval[-17], trueval[-99], newrs[100]

In [ ]:
print(np.diff(old_trueval)[np.where(np.diff(old_newco)<0)],max(np.where(old_newco<np.mean(old_newco[:6])+np.std(old_newco[:6]))[0]))
print(np.diff(trueval)[np.where(np.diff(correction)<0)])
print(old_newco[0:6], np.mean(old_newco[:6]),  np.std(old_newco[:6]))
print(old_newco[0]+0.0003149, old_newco[8])
plt.plot(old_newco[:37]-0.11, "+")
plt.plot(np.diff(old_newco)[:37])

In [ ]:
np.polyfit(trueval, correction-old_newco,1, cov=True), np.sqrt(np.diag(np.polyfit(trueval, correction-old_newco,1, cov=True)[1])), pearsonr(trueval, correction-old_newco)

In [ ]:
from scipy.io import loadmat
loadmat("/home/tani/Documents/nonLinear/MS_fMRI.mat")['MS_fMRI'].shape

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(-0.5 * np.log(1 - (np.arange(400) * incre/2) ** 2),np.arange(400) * incre/2, )
plt.plot( -0.5 * np.log(1 - newrs ** 2),newrs, "o", markersize=1)
plt.gca().set_aspect("equal")

In [ ]:
t = symbols("t")
m = -0.5 * log(1 - t ** 2)
dl = sqrt(diff(t,t)**2+diff(m,t)**2)
display(simplify(dl))
integrate(dl, (t,0,0.995))

In [ ]:
dl_n=lambdify(t, dl)

In [ ]:
eps=1e-7
length = 0
num_steps=70
interv = 2.8223199056193393/num_steps
newrs=np.zeros(num_steps)
j=1
for i in range(int(0.995*1e7)):
    length+=dl_n(i*eps)*1e-7
    if length>interv*j:
        newrs[j]=i*eps
        j+=1
print(length, j)
print(newrs)

In [ ]:
plt.plot(np.sqrt(np.diff( -0.5 * np.log(1 - newrs ** 2))**2+np.diff(newrs)**2))
np.diff(newrs)[:10],np.diff(newrs)[-10:], np.diff(-0.5 * np.log(1 - newrs ** 2))[:10]

In [ ]:
newrs

## Expected effect of remapping

In [ ]:
from scipy.stats import pearsonr
from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
rho=0.7
means = 0, 0
correlations = [[1, rho], [rho, 1]]
n_samples = 400
n_iterations = 75000
effective = np.empty(n_iterations)
for j in tqdm(range(n_iterations)):
    points = np.random.multivariate_normal(means, correlations, n_samples).T.copy()
    effective[j] = pearsonr(*points)[0]

sigma = 1/np.sqrt(n_samples-3)
mi = 1/2*np.log((1+rho)/(1-rho))
emme = np.linspace(mi-6*sigma, mi+6*sigma,10001)
qwe = norm(mi, sigma)
prob = qwe.pdf(emme)
erre = (np.exp(2*emme)-1)/(np.exp(2*emme)+1)
bsiz=np.diff(erre)
centval=prob[:-1]+np.diff(prob)/2
renorm=bsiz@centval

In [ ]:
Q=plt.hist(effective, bins=300,density=True)
plt.plot(erre, prob/renorm, "--r")
plt.xlabel("r")
plt.ylabel("prob")
plt.show()

In [ ]:
centerre = erre[:-1]+np.diff(erre)/2
new_weights=centval*bsiz/renorm
wMean = np.average(centerre, weights=new_weights)
wDev = np.sqrt(np.sum((centerre-wMean)**2*new_weights))
print(wMean, wDev, np.mean(effective), np.std(effective), np.mean(effective)-wMean)
centerre = erre[:-1]+np.diff(erre)/2
new_weights=centval*bsiz/renorm
wWMean = np.average(erre, weights=prob)
wWDev = np.sqrt(np.sum((erre-wWMean)**2*prob)/renorm)
print(wWMean, wDev, np.mean(effective), np.std(effective))
print(np.quantile(effective,[.025,.975]))

In [ ]:
from scipy.stats import skew, skewtest
skew(effective), skewtest(effective)

In [ ]:
from scipy.stats import ttest_1samp
ttest_1samp(effective, wMean)

In [ ]:
"mode", erre[np.argmax(prob)], "median", centerre[np.argmin(np.abs(np.cumsum(new_weights)-0.5))], np.median(effective)

In [ ]:
transEmme=(np.exp(2*emme)-1)/(np.exp(2*emme)+1)
plt.plot(emme,transEmme )
plt.plot([min(transEmme),max(transEmme)], [min(transEmme),max(transEmme)], ":k")
plt.show()

In [ ]:
from scipy.stats import norm
qwe = norm(mi, sigma)
twistedEffective=0.5*np.log((1+effective)/(1-effective))
plt.hist(twistedEffective, bins=150, density=True)
plt.plot(np.linspace(0.4,0.7,500), qwe.pdf(np.linspace(0.4,0.7,500)))
skew(twistedEffective), skewtest(twistedEffective),ttest_1samp(twistedEffective, mi), mi, np.mean(twistedEffective), sigma, np.std(twistedEffective)

In [ ]:
from scipy.stats import multivariate_normal
rho=0.1
correlations = [[1, rho], [rho, 1]]
mulv = multivariate_normal(means, correlations)
singv= norm(0,1)

In [ ]:
def effect (x, y, pt, r, N):
    dy = pt-y
    dx = pt-x
    num = r + (x*dy + y*dx + dx*dy*(1-1/N))/N
    den = np.sqrt( (1+ (2*dx*x+dx**2*(1-1/N))/N)*(1+(2*dy*y+dy**2*(1-1/N))/N) )
    return num/den-r

In [ ]:
mv_cdf=np.zeros([200,200])
exp_gain=np.zeros([200,200])
n_punti=400
pt=singv.ppf((n_punti-0.5)/n_punti)
for i,a in enumerate(np.linspace(0,5,200)):
    for j,b in enumerate(np.linspace(0,5,200)):
        mv_cdf[i,j]=np.exp(n_punti*mulv.logcdf([a,b]))*mulv.pdf([a,b])
        exp_gain[i,j]=effect (a, b, pt, rho, n_punti)
mv_cdf/=np.sum(mv_cdf)/(199/5)**2
drpt = pt*200/5
plt.scatter(drpt,drpt)
plt.imshow(mv_cdf)
plt.gca().invert_yaxis()
plt.colorbar()
plt.show()

plt.imshow(exp_gain*mv_cdf)
plt.gca().invert_yaxis()
plt.colorbar()
plt.scatter(drpt,drpt)
plt.show()

plt.imshow(exp_gain)
plt.gca().invert_yaxis()
plt.colorbar()
plt.scatter(drpt,drpt)
plt.show()

## Effect of different preparations

In [ ]:
def linnoise (x, a):
    return (a*x+np.random.rand(*x.shape)+1)

def gannoise (x, a):
    return (a*x+np.random.normal(0,1,x.shape)+1)

def normalise(vec):
    rv = norm(0, 1)
    return rv.ppf((np.argsort(np.argsort(vec)) + 0.5) / len(vec))

x=np.random.rand(400)
y=linnoise(x,1)
plt.scatter(x,y, label="Linear Noise")
gx=np.random.rand(400)
gy=gannoise(gx,3.5)
plt.scatter(gx,gy, label="Gaussian Noise", alpha=0.7)
plt.title(f"Lin: {pearsonr(x,y)[0]:.4} - Gauss: {pearsonr(gx,gy)[0]:.4}")
plt.suptitle("Before gaussianisation")
plt.legend()
plt.show()

nx=normalise(x)
ny=normalise(y)
ngx=normalise(gx)
ngy=normalise(gy)
plt.scatter(nx,ny, label="Linear Noise")
plt.scatter(ngx,ngy, label="Gaussian Noise", alpha=0.7)
plt.title(f"Lin: {pearsonr(nx,ny)[0]:.4} - Gauss: {pearsonr(ngx,ngy)[0]:.4}")
plt.suptitle("After gaussianisation")
plt.legend()
plt.show()

bx = np.concatenate([np.random.normal(-3,1,100),np.random.normal(3,1,100)])
by = bx+np.random.normal(0,1,200)#np.random.rand(*bx.shape)-0.5#
nbx = normalise(bx)
nby = normalise(by)
plt.scatter(bx,by, label="Before")
plt.scatter(nbx,nby, label="After", alpha=0.7)
plt.title(f"Before: {pearsonr(bx,by)[0]:.4} - After: {pearsonr(nbx,nby)[0]:.4}")
plt.suptitle("Bimodal with gaussian noise")
plt.legend(title="gaussianisation")
plt.show()



In [ ]:
from support import surrogate
bx = np.concatenate([np.random.normal(-3,1,500),np.random.normal(3,1,500)])
by = bx+np.random.normal(0,1,1000)#np.random.rand(*bx.shape)-0.5#
nbx = normalise(bx)
nby = normalise(by)
fix, ax = plt.subplots(2,2,sharex=True, sharey=True)
plt.sca(ax[0,0])
plt.scatter(bx, by, label=f"{pearsonr(bx,by)[0]:.4}")
plt.title("Original")
plt.legend()


plt.sca(ax[0,1])
plt.scatter(nbx, nby, label=f"{pearsonr(nbx,nby)[0]:.4}")
snb=surrogate(np.array([nbx, nby]).T)
plt.scatter(*snb.T, label=f"s:{pearsonr(*snb.T)[0]:.4}")
plt.title("Gaussianised")
plt.legend()


plt.sca(ax[1,0])
lbx=(bx-np.mean(bx))/np.sqrt(np.std(bx))
lby=(by-np.mean(by))/np.sqrt(np.std(by))
plt.scatter(lbx, lby, label=f"{pearsonr(lbx,lby)[0]:.4}")
slb=surrogate(np.array([lbx, lby]).T)
plt.scatter(*slb.T, label=f"s:{pearsonr(*slb.T)[0]:.4}")
plt.title("Linear normalisation")
plt.legend()


plt.sca(ax[1,1])
plt.scatter(bx, by, label=f"{pearsonr(bx,by)[0]:.4}")
sb=surrogate(np.array([bx, by]).T)
plt.scatter(*sb.T, label=f"s:{pearsonr(*sb.T)[0]:.4}")
plt.title("No transform")
plt.legend()
plt.show()

In [ ]:
plt.hist(bx)
plt.hist(snb.T[0], alpha=0.5, label="gauss,sh")
plt.hist(slb.T[0]*np.sqrt(np.std(by)), alpha=0.5, label="linear,sh")
plt.hist(sb.T[0], alpha=0.5, label="nothing,sh")
plt.legend()

In [ ]:
def moving_average(a, n=3) :
    ret = np.cumsum(a, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n - 1:] / n
fig, ax = plt.subplots(2,1,squeeze=True,figsize=(10,16))
for seq, nam in [(bx, "raw"),(sb.T[0], "nothing"),(snb.T[0], "gauss"),(slb.T[0], "linear")]:
    ft=np.fft.rfft(seq)
    plt.sca(ax[0])
    plt.hist(np.angle(ft), label=nam, alpha=0.5)
    plt.sca(ax[1])
    #plt.hist(np.log(np.absolute(ft)), label=nam, alpha=0.5, bins="auto")
    plt.plot(moving_average(np.absolute(ft),10)/(np.sqrt(3.1766164689316008) if nam=="nothing" else 1), label=nam)
plt.sca(ax[0])
plt.legend()
plt.sca(ax[1])
plt.legend()
plt.yscale("symlog")
plt.show()



In [ ]:
np.std(bx)

In [ ]:
px = np.concatenate([np.random.normal(-3,1,2500),np.random.normal(3,1,2500)])[...,np.newaxis].T
px2 = surrogate(px.T).T
phi=np.angle(np.fft.rfft(px))
phi2=np.angle(np.fft.rfft(px2))
plt.scatter(phi,np.roll(phi,1), color="k", s=5)
plt.scatter(phi2,np.roll(phi2,1), color="r", s=5)
plt.xlim([-np.pi,np.pi])
plt.ylim([-np.pi,np.pi])
plt.show()

In [ ]:
from sympy import *
x, a, b = symbols("x a b", real=True, positive=True)
k = symbols("k")
f=abs(x)*exp(-a*x**2)
f

In [ ]:
integrate(f,(x,-np.inf, np.inf))

In [ ]:
Derivative(tan(x),x,evaluate=True)

In [ ]:
x, a, b = symbols("x a b", real=True)#, positive=True)
k = symbols("k")
g=a/sqrt(x)*exp(-b*x)
g

In [ ]:
fourier_transform(g,x,k)

In [ ]:
x, B, S_r, S_i , r = symbols("x B S_r S_i r", real=True)
M = 
h = exp(-M*cos(x))
h
besseli(0,x)

In [ ]:
integrate(h, (x,0,np.pi))

In [ ]:
s, r = symbols("s r")
f = 1/s**2*exp(-r**2/2/s**2)*r
integrate(f, (r,0,np.inf))

In [ ]:
_bx = np.concatenate([np.random.normal(-3,1,50000),np.random.normal(3,1,50000)])
alpha=+0.5
alpha2=-0.4
bx=np.empty_like(_bx)
bx[0]=_bx[0]
for j in range(1,len(bx)):
    if j<5:
        bx[j]=bx[j-1]*alpha+_bx[j]
    else:
        bx[j]=bx[j-1]*alpha+bx[j-5]*alpha2+_bx[j]
ft=np.fft.rfft(bx)
phi=np.angle(np.fft.rfft(ft))
plt.scatter(phi,np.roll(phi,1), color="k", s=5, alpha=0.5, label=1)
plt.xlim([-np.pi,np.pi])
plt.ylim([-np.pi,np.pi])
plt.legend()
plt.show()
plt.scatter(phi,np.roll(phi,2), color="r", s=5, alpha=0.5, label=2)
plt.xlim([-np.pi,np.pi])
plt.ylim([-np.pi,np.pi])
plt.legend()
plt.show()
plt.scatter(phi,np.roll(phi,5), color="b", s=5, alpha=0.5, label=5)
plt.xlim([-np.pi,np.pi])
plt.ylim([-np.pi,np.pi])
plt.legend()
plt.show()
plt.scatter(phi,np.roll(phi,10), color="g", s=5, alpha=0.5, label=10)
plt.xlim([-np.pi,np.pi])
plt.ylim([-np.pi,np.pi])
plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(16,7))
def distr(r, s):
    return 1/s**2*np.exp(-r**2/2/s**2)*r
ft=np.fft.rfft(bx)
plt.sca(ax[0])
q=plt.hist(np.absolute(ft), label=4-i, alpha=0.5, bins="auto", density=True)
#plt.hist(np.log(np.absolute(ft)), label=nam, alpha=0.5, bins="auto")
sig=np.sqrt(len(bx)/2)
plt.plot(np.arange(1000), distr(np.arange(1000), sig), color=q[2][0].get_facecolor())
plt.xlim((0,1000))
plt.sca(ax[1])
q=plt.hist(np.angle(ft), label=4-i, alpha=0.5, bins="auto", density=True)
plt.legend()
plt.show()


In [ ]:
fsurr = ft*np.exp(2 * np.pi * np.random.rand(int(bx.shape[0] / 2 + 1)) * 1.0j)
sx = np.fft.irfft(fsurr, n=bx.shape[0])
plt.hist(bx, alpha=0.5, bins="auto", density=True)
plt.hist(sx, alpha=0.5, bins="auto", density=True)
plt.show()

In [ ]:
from scipy.stats import rayleigh

rv=rayleigh(0,sig)
vec = np.absolute(ft)
rvec=rv.ppf((np.argsort(np.argsort(vec)) + 0.5) / len(vec))

plt.hist(vec, label=4-i, alpha=0.5, bins=50, density=True, range=(0,1000))
q=plt.hist(rvec, label=4-i, alpha=0.5, bins=50, density=True)
#plt.hist(np.log(np.absolute(ft)), label=nam, alpha=0.5, bins="auto")
sig=np.sqrt(len(bx)/2)
plt.plot(np.arange(1000), distr(np.arange(1000), sig), color=q[2][0].get_facecolor())
plt.xlim((0,1000))
plt.show()
fakeFT=rvec*np.exp(2 * np.pi * np.random.rand(int(bx.shape[0] / 2 + 1)) * 1.0j)
tx = np.fft.irfft(fakeFT, n=bx.shape[0])
fsurr = ft*np.exp(2 * np.pi * np.random.rand(int(bx.shape[0] / 2 + 1)) * 1.0j)
sx = np.fft.irfft(fsurr, n=bx.shape[0])
plt.hist(bx, alpha=0.5, bins="auto", density=True)
plt.hist(sx, alpha=0.5, bins="auto", density=True)
plt.hist(tx, alpha=0.5, bins=40, density=True)
plt.show()

In [ ]:
mycos = np.cos(2*np.pi*np.random.rand(10000000))*rayleigh.rvs(0, sig, size=10000000)/sig
plt.hist(mycos, bins="auto", density=True)
plt.show()

In [ ]:
np.std(mycos), sig

In [ ]:
wx = np.arange(1001)/1000
plt.plot(wx, np.power(1*4*wx*(1-wx),1/30))
plt.show()

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(16,7))
plt.sca(ax[0])
plt.plot(np.arange(len(ft)-300)/len(ft),moving_average(np.absolute(ft), 301))
plt.sca(ax[1])
plt.plot(bx)
plt.show()

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(16,7))
for i in range(4):
    bx = np.concatenate([np.random.normal(-3,1,1000*(4-i)),np.random.normal(3,1,1000*(4-i))])
    ft=np.fft.rfft(bx)
    plt.sca(ax[0])
    q=plt.hist(np.absolute(ft), label=4-i, alpha=0.5, bins="auto", density=True)
    #plt.hist(np.log(np.absolute(ft)), label=nam, alpha=0.5, bins="auto")
    sig=np.sqrt(len(bx)/2)
    plt.plot(np.arange(175), distr(np.arange(175), sig), color=q[2][0].get_facecolor())
    plt.xlim((0,200))
    plt.sca(ax[1])
    q=plt.hist(np.angle(ft), label=4-i, alpha=0.5, bins="auto", density=True)
plt.legend()
plt.show()

In [ ]:
def distr(r, s):
    return 1/s**2*np.exp(-r**2/2/s**2)*r
fig, ax = plt.subplots(2,2,figsize=(16,16))
alpha=+0.5
alpha2=+0.4
for i in range(4):
    _seq=np.random.normal(0,1,1000*(4-i))
    seq=np.empty_like(_seq)
    seq[0]=_seq[0]
    for j in range(1,1000*(4-i)):
        if j<5:
            seq[j]=seq[j-1]*alpha+_seq[j]
        else:
            seq[j]=seq[j-1]*alpha+seq[j-5]*alpha2+_seq[j]
    seq-=np.mean(seq)
    ft=np.fft.rfft(seq)
    plt.sca(ax[0,0])
    plt.hist(np.angle(ft), label=4-i, alpha=0.5, density=True)
    plt.sca(ax[0,1])
    plt.plot(np.angle(ft), label=4-i)
    plt.sca(ax[1,0])
    q=plt.hist(np.absolute(ft), label=4-i, alpha=0.5, bins="auto", density=True)
    #plt.hist(np.log(np.absolute(ft)), label=nam, alpha=0.5, bins="auto")
    sig=np.sqrt(500*(4-i))
    plt.plot(np.arange(175), distr(np.arange(175), sig), color=q[2][0].get_facecolor())
    plt.sca(ax[1,1])
    plt.plot(np.arange(len(ft)-10*(4-i)+1)/len(ft),moving_average(np.absolute(ft),10*(4-i)), label=4-i)
    plt.plot()
plt.sca(ax[0,0])
plt.legend()
plt.sca(ax[1,1])
plt.legend()
# plt.yscale("symlog")
plt.show()

In [ ]:
_N=100
def sigma_ik (k, _N):
    return -np.sin(2*np.pi*k*(2-1/_N))/4/np.sin(2*np.pi*k/_N)+_N/2-1/4
for _N in range(100,500,100):
    p=plt.plot(np.arange(0,_N//2+2),sigma_ik(np.arange(0,_N//2+2), _N), label=_N)
    s=np.zeros(_N//2+2)
    for i in range(_N//2+2):
        s[i]=np.sum(np.sin(2*np.pi*i/_N*np.arange(0,_N))**2)
    plt.plot(np.arange(0,_N//2+2),s, color=p[0].get_color(), ls="--")
plt.legend()
plt.show()

In [ ]:
from scipy.special import i0, i0e

plt.plot(np.linspace(0,100,1000),i0e(np.linspace(0,100,1000)))
plt.plot([1,100],0.4*np.array([1,0.1]))
plt.yscale("log")
plt.xscale("log")

## Phase scrambling

In [ ]:
from scipy.stats import vonmises
samples=10000
vm = vonmises(0.1)
x=np.random.random(samples)*np.pi*2
p=vonmises.rvs(1,size=samples)+np.pi
plt.hist(p, bins=np.linspace(0,2*np.pi,16))
plt.hist(x, bins=np.linspace(0,2*np.pi,16), alpha=0.5)
y=(p+x)%(2*np.pi)
plt.hist(y, bins=np.linspace(0,2*np.pi,16), alpha=0.5)
plt.show()

In [ ]:
import numpy as np

from scipy.stats import vonmises

import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1)
kappa = 0.1

mean, var, skew, kurt = vonmises.stats(kappa, moments='mvsk')
x = np.linspace(vonmises.ppf(0.01, kappa), vonmises.ppf(0.99, kappa), 100)

ax.plot(x, vonmises.pdf(x, kappa),'r-', lw=5, alpha=0.6, label='vonmises pdf')
plt.xlim([-np.pi,np.pi])
plt.ylim(bottom=0)

## alternative gaussianisation 

In [ ]:
import pandas as pd
def normalise(vec):
    rv = norm(0, 1)
    return rv.ppf((np.argsort(np.argsort(vec)) + 0.5) / len(vec))
def normalise2(vec):
    rv = np.sort(np.random.normal(0, 1, len(vec)))
    return rv[np.argsort(np.argsort(vec))]
results=[]

for i in range(10000):
    pts=np.random.multivariate_normal(np.zeros(2), [[1,.7],[.7,1]], 500)

    ptsN = np.zeros_like(pts)
    for i in range(pts.shape[1]):
        ptsN[:, i] = normalise(pts[:, i])

    ptsN2 = np.zeros_like(pts)
    for i in range(pts.shape[1]):
        ptsN2[:, i] = normalise2(pts[:, i])

    results.append([pearsonr(*pts.T)[0],pearsonr(*ptsN.T)[0],pearsonr(*ptsN2.T)[0],])
res_df = pd.DataFrame(results, columns=["r", "rN", "rN2"])
res_df.hist()

In [ ]:
rv = np.random.normal(0, 1, 100)
plt.scatter(rv, normalise(rv))
plt.scatter(rv, normalise2(rv))
res_df.describe()

In [ ]:
plt.hist(np.random.normal(0,1,10000)**2, bins="auto")
plt.yscale("log")
plt.show()